In [ ]:
print("""
CERN Electron Collision Sensor Data Analysis
Notebook 8: Sensor Purging
Objectives:
1. Simulate sensor corruption/noise in physical measurements
2. Build a physics-based anomaly detection algorithm
3. Purge anomalous events and evaluate recovery of downstream performance
""")

In [ ]:
# Import required libraries
import warnings
warnings.filterwarnings('ignore')
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
print("Libraries loaded successfully")

In [ ]:
# Load the best-performing model bundle
model_bundle = joblib.load('../models/best_model_bundle.pkl')
best_model = model_bundle['model']
selected_features = model_bundle['selected_features']
requires_scaling = model_bundle['requires_scaling']
scaler = model_bundle['scaler']
print(f"Loaded model: {model_bundle['model_name']}")

In [ ]:
# Load full engineered dataset and prepare healthy test set
df = pd.read_csv('../data/processed/dielectron_engineered.csv')
df.columns = df.columns.str.strip()
_, df_test = train_test_split(df, test_size=0.20, random_state=42)
df_test = df_test.copy()
print(f"Healthy test dataset shape: {df_test.shape}")

In [ ]:
# Simulate sensor failure by injecting corruption
# We zero out E2 and pt2 for 5% of events to represent dead channels/calorimeter failure
np.random.seed(42)
corrupted_indices = np.random.choice(df_test.index, size=int(len(df_test) * 0.05), replace=False)
df_test_corrupted = df_test.copy()
df_test_corrupted.loc[corrupted_indices, 'E2'] = 0.0
df_test_corrupted.loc[corrupted_indices, 'pt2'] = 0.0
# Recalculate any features that depend on E2 and pt2 in the corrupted data
df_test_corrupted.loc[corrupted_indices, 'Total_Energy'] = df_test_corrupted.loc[corrupted_indices, 'E1']
df_test_corrupted.loc[corrupted_indices, 'Energy_Difference'] = df_test_corrupted.loc[corrupted_indices, 'E1']
df_test_corrupted.loc[corrupted_indices, 'Energy_Ratio'] = 0.0
df_test_corrupted.loc[corrupted_indices, 'Total_PT'] = df_test_corrupted.loc[corrupted_indices, 'pt1']
df_test_corrupted.loc[corrupted_indices, 'PT_Difference'] = df_test_corrupted.loc[corrupted_indices, 'pt1']
df_test_corrupted.loc[corrupted_indices, 'PT_Ratio'] = 0.0
print(f"Injected dead-sensor failures into {len(corrupted_indices)} events (5% of test data)")

In [ ]:
# Implement physics-based anomaly detection
# An event is flagged if energy E2 is exactly zero or if E2 deviates significantly
# from the momentum magnitude: E_reconstructed = sqrt(px^2 + py^2 + pz^2)
E2_calculated = np.sqrt(df_test_corrupted['px2']**2 + df_test_corrupted['py2']**2 + df_test_corrupted['pz2']**2)
# Calculate the reconstruction error. Deviations between measured E2 and sqrt(px2^2 + py2^2 + pz2^2) validate the relativistic energy-momentum relation. A high error indicates sensor degradation or failure, and purging these events prevents the model from learning from corrupted sensor data.
reconstruction_error = np.abs(df_test_corrupted['E2'] - E2_calculated)
anomaly_mask = (df_test_corrupted['E2'] == 0.0) | (reconstruction_error > 0.5)
detected_anomalies_indices = df_test_corrupted[anomaly_mask].index
print(f"Identified {len(detected_anomalies_indices)} anomalies using physics validation checks")

In [ ]:
# Purge identified anomalies from the dataset
df_test_purged = df_test_corrupted.drop(index=detected_anomalies_indices)
print(f"Purged anomalous records. Remaining records: {len(df_test_purged)}")

In [ ]:
# Helper function to scale and predict
def get_predictions(model, data_df):
    X_data = data_df[selected_features]
    if requires_scaling and scaler is not None:
        X_data_scaled = scaler.transform(X_data)
        X_data_model = pd.DataFrame(X_data_scaled, columns=X_data.columns, index=X_data.index)
    else:
        X_data_model = X_data
    return model.predict(X_data_model)
print("Prediction helper function defined")

In [ ]:
# Make predictions on healthy, corrupted, and purged test sets
pred_healthy = get_predictions(best_model, df_test)
pred_corrupted = get_predictions(best_model, df_test_corrupted)
pred_purged = get_predictions(best_model, df_test_purged)
y_test_purged = df_test_purged['M']
print("Predictions computed across all subsets")

In [ ]:
# Calculate evaluation metrics
rmse_healthy = np.sqrt(mean_squared_error(df_test['M'], pred_healthy))
r2_healthy = r2_score(df_test['M'], pred_healthy)
rmse_corrupted = np.sqrt(mean_squared_error(df_test_corrupted['M'], pred_corrupted))
r2_corrupted = r2_score(df_test_corrupted['M'], pred_corrupted)
rmse_purged = np.sqrt(mean_squared_error(y_test_purged, pred_purged))
r2_purged = r2_score(y_test_purged, pred_purged)

metrics_df = pd.DataFrame({
    'Dataset': ['Healthy', 'Corrupted (5%)', 'Purged (Cleaned)'],
    'RMSE': [rmse_healthy, rmse_corrupted, rmse_purged],
    'R2': [r2_healthy, r2_corrupted, r2_purged]
})
display(metrics_df)

In [ ]:
# Plot downstream prediction performance comparison
plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_df, x='Dataset', y='RMSE', palette='Oranges_d')
plt.title("Impact of Sensor Failures and Purging on Downstream Model RMSE")
plt.ylabel("RMSE (GeV)")
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Save results to outputs
os.makedirs('../outputs', exist_ok=True)
metrics_df.to_csv('../outputs/sensor_purging_results.csv', index=False)
print("Sensor purging results saved successfully")